1.Environment Setup

In [ ]:
!pip install pandas nltk
!pip install spacy
!python -m spacy download en_core_web_sm
!pip install pandas numpy matplotlib seaborn scikit-learn nltk transformers torch vaderSentiment
!pip install transformers[torch]` or `!pip install 'accelerate>=0.26.0'

In [ ]:
import re, sys, joblib,	os
import numpy as np
import pandas as pd
import spacy as sp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support


import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import os 
from nltk.tokenize import word_tokenize

from nltk.stem import PorterStemmer

from nltk.stem import WordNetLemmatizer
import nltk

from nltk.corpus import wordnet




In [ ]:
import sys
print(sys.executable)
import sys
!{sys.executable} -m pip install -U spacy
!{sys.executable} -m spacy download en_core_web_sm

print("Python:", sys.executable)
print("Torch:", torch.__version__)

1.2 hyperparameters / constants(Ziqi)

In [ ]:
CSV_PATH = "dataset_A_news_full_10500.csv"
TEXT_COL = "title"
LABEL_COL = "classes_str"

RANDOM_SEED = 42
TEST_SIZE = 0.2

TFIDF_MAX_FEATURES = 5000

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64
NUM_EPOCHS = 3
TRAIN_BS = 4
EVAL_BS = 4
GRAD_ACCUM = 4

OUTPUT_DIR = "./results_integrated"
SAVE_MODEL_DIR = "./my_bert_model_integrated"
LABEL_ENCODER_PATH = "label_encoder.pkl"

1.3 NLTK Download

In [ ]:
NLTK_DIR = os.path.expanduser("~/nltk_data")
os.makedirs(NLTK_DIR, exist_ok=True)
if NLTK_DIR not in nltk.data.path:
nltk.data.path.append(NLTK_DIR)
nltk.download("punkt", download_dir=NLTK_DIR)

1.4 Text preprocessing pipeline+Tokens Clean

In [ ]:
df = pd.read_csv(r"C:\Users\Kiraa\OneDrive\Desktop\dataset_A_news_full_10500.csv")
df = df.dropna(subset=["classes_str"]).copy()

texts = df["classes_str"].astype(str)
#字段名 classes_str 听起来像“类别字符串”，但你这里把它当文本在处理；如果它确实是类别而不是正文，那这段就处理错列了。
df["tokens"] = texts.apply(word_tokenize)
df["tokens_clean"] = df["tokens"].apply(
lambda toks: [t.lower() for t in toks if t.isalpha()]
)

stemmer = PorterStemmer()
df["stems"] = df["tokens_clean"].apply(lambda toks: [stemmer.stem(t) for t in toks])

df[["classes_str", "tokens", "tokens_clean", "stems"]].head()

2.1 POS Tagging

In [ ]:
nltk.download("averaged_perceptron_tagger", download_dir=NLTK_DIR)
nltk.download("averaged_perceptron_tagger_eng", download_dir=NLTK_DIR)
df["pos_tags"] = df["lemmas"].apply(
lambda toks: nltk.pos_tag(toks, tagset=None)
)

2.2 Lemmatization

In [ ]:
nltk.download("wordnet", download_dir=NLTK_DIR)
nltk.download("omw-1.4", download_dir=NLTK_DIR)
lemmatizer = WordNetLemmatizer()

df["lemmas"] = df["tokens_clean"].apply(
lambda toks: [lemmatizer.lemmatize(t) for t in toks]
)

df[["tokens_clean", "lemmas"]].head()

2.3 Penn Treebank POS-> WordNet POS

In [ ]:
def penn_to_wordnet_pos(penn_tag: str):
    if not penn_tag:
        return wordnet.NOUN

    c = penn_tag[0]
    if c == "J":
        return wordnet.ADJ
    if c == "V":
        return wordnet.VERB
    if c == "N":
        return wordnet.NOUN
    if c == "R":
        return wordnet.ADV
    return wordnet.NOUN


2.4 POS Tagging+Lemmatization

In [ ]:
def pos_tag_tokens(tokens):
    # tokens: List[str]
    return nltk.pos_tag(tokens)  # -> List[(word, tag)]

def lemmatize_with_pos(tokens):
    tagged = pos_tag_tokens(tokens)
    lemmas = [lemmatizer.lemmatize(w, pos=penn_to_wordnet_pos(t)) for w, t in tagged]
    return tagged, lemmas

# 生成两列：pos_tags 和 lemmas_pos
df[["pos_tags", "lemmas_pos"]] = df["tokens_clean"].apply(
    lambda toks: pd.Series(lemmatize_with_pos(toks))
)

# 预览
df[["tokens_clean", "pos_tags", "lemmas_pos"]].head()

2.5 Use spaCy do POS tagging 

In [ ]:
import sys
print(sys.executable)

sys.executable
from collections import Counter

nlp = spacy.load("en_core_web_sm")
df = df.dropna(subset=["classes_str"]).copy()
df["classes_str"] = df["classes_str"].astype(str)

if "lemmas" not in df.columns:
    if "tokens_clean" not in df.columns:
        raise ValueError("Need df['tokens_clean'] or df['lemmas'] before POS tagging.")
    df["lemmas"] = df["tokens_clean"]

def spacy_pos(tokens):
    doc = nlp(" ".join(tokens))
    return [(t.text, t.pos_) for t in doc]

df["pos_tags"] = df["lemmas"].apply(spacy_pos)

df["nouns"] = df["pos_tags"].apply(
    lambda tags: [w for w, pos in tags if pos in ("NOUN", "PROPN")]
)

df[["classes_str", "pos_tags", "nouns"]].head()
#直接对原始文本 classes_str 做 nlp(text)，然后从 token.lemma_、token.pos_ 里取（spaCy 自带 lemma）。

2.6 Chunking

In [ ]:
def extract_noun_phrases(text):
    doc = nlp(text)
    return [chunk.text.lower() for chunk in doc.noun_chunks if any(c.isalnum() for c in chunk.text)]

df["noun_phrases"] = df["classes_str"].apply(extract_noun_phrases)

df[["classes_str", "noun_phrases"]].head(10)

3.1 Data column name validation+cleaning+overview

In [ ]:
df.columns = df.columns.astype(str).str.strip()

need = {TEXT_COL, LABEL_COL}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {list(df.columns)}")

df = df.dropna(subset=[LABEL_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
df[LABEL_COL] = df[LABEL_COL].astype(str).fillna("")

print("Rows:", len(df))
print("Num classes:", df[LABEL_COL].nunique())

: 

3.2 Pretrain TF-IDF

In [ ]:
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

def preprocess_for_tfidf(text: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", str(text).lower())
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df[TEXT_COL].apply(preprocess_for_tfidf)
#列名是否用反/“保留数字”是否符合你的任务/这个清洗对英语缩写/否定不友好/可以把“去停用词”交给 TfidfVectorizer

3.2.1Training set/validation set split

In [ ]:
idx = np.arange(len(df))
label_counts = df[LABEL_COL].value_counts()
df["_label_for_split"] = df[LABEL_COL].where(df[LABEL_COL].map(label_counts) >= 2, "OTHER_RARE")

train_idx, val_idx = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df["_label_for_split"]
)

df_train = df.iloc[train_idx].copy()
df_val   = df.iloc[val_idx].copy()

print("Train:", len(df_train), "Val:", len(df_val))
print("Rare merged into OTHER_RARE:", (df["_label_for_split"] == "OTHER_RARE").sum())

3.3 Baseline->TF-IDF->Multinomial Naive Bayes

In [ ]:
print("\n" + "="*60)
print("A) Baseline: TF-IDF + MultinomialNB")
print("="*60)

tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES)
X_train_tfidf = tfidf.fit_transform(df_train["clean_text"])
X_val_tfidf   = tfidf.transform(df_val["clean_text"])

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, df_train[LABEL_COL])

nb_pred = nb_model.predict(X_val_tfidf)
print("NB Accuracy:", accuracy_score(df_val[LABEL_COL], nb_pred))
print(classification_report(df_val[LABEL_COL], nb_pred))

df_val["nb_pred"] = nb_pred

3.4用 Hugging Face Transformers 的 Trainer 训练一个 distilbert-base-uncased 的新闻标题分类模型（多分类），流程是：装依赖 → 标签编码 → tokenizer → Dataset 包装 → 加载预训练模型（加分类头）→ 配置训练参数 → 训练 → 验证评估 → 输出 classification report

In [ ]:
le = LabelEncoder()
le.fit(df[LABEL_COL])

y_train_int = le.transform(df_train[LABEL_COL])
y_val_int   = le.transform(df_val[LABEL_COL])

num_labels = len(le.classes_)
print("Number of classes:", num_labels)

# -----------------------------
# 2) Tokenizer
# -----------------------------
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# -----------------------------
# 3) Dataset wrapper
# -----------------------------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = NewsDataset(df_train["title"], y_train_int, tokenizer)
val_dataset   = NewsDataset(df_val["title"],   y_val_int,   tokenizer)

# -----------------------------
# 4) Model
# -----------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

# -----------------------------
# 5) Metrics
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

# -----------------------------
# 6) Training arguments (CPU-safe)
# -----------------------------
training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",          # ✅ CORRECT
    save_strategy="epoch",
    report_to="none",
    use_cpu=True,
    fp16=False
)


# -----------------------------
# 7) Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# -----------------------------
# 8) Train
# -----------------------------
trainer.train()

# -----------------------------
# 9) Evaluate
# -----------------------------
results = trainer.evaluate()
print("Validation results:", results)

preds = np.argmax(trainer.predict(val_dataset).predictions, axis=-1)
labels = list(range(len(le.classes_)))  # ensure full label set

print(classification_report(
    y_val_int,
    preds,
    labels=labels,
    target_names=le.classes_,
    zero_division=0
))

3.5 做 DistilBERT 在验证集上的更完整评估 + 可视化误差分析：不仅算 macro/weighted F1，还打印完整分类报告、画“Top-N 类别”的混淆矩阵热力图，并列出最常见的“混淆对”（A 类经常被预测成 B 类）。

In [ ]:
# 1) Predictions on validation set
pred_out = trainer.predict(val_dataset)
val_preds = np.argmax(pred_out.predictions, axis=-1)
val_true  = np.array(y_val_int)

# 2) Summary metrics (useful for many-class, imbalanced data)
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    val_true, val_preds, average="macro", zero_division=0
)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(
    val_true, val_preds, average="weighted", zero_division=0
)

print("\n" + "="*60)
print("PART C) DistilBERT Evaluation Summary")
print("="*60)
print(f"Macro F1:     {f1_macro:.4f}")
print(f"Weighted F1:  {f1_w:.4f}")

# 3) Full classification report (safe: pass labels explicitly)
labels_all = list(range(len(le.classes_)))
print("\nClassification report (may be long):")
print(classification_report(
    val_true,
    val_preds,
    labels=labels_all,
    target_names=le.classes_,
    zero_division=0
))

# 4) Confusion matrix: show only TOP-N most frequent classes in val_true (readable)
TOP_N = 25  # change to 15/20/30 depending on readability

val_counts = pd.Series(val_true).value_counts()
top_labels = val_counts.head(TOP_N).index.tolist()

cm_top = confusion_matrix(val_true, val_preds, labels=top_labels)
top_names = [le.classes_[i] for i in top_labels]

plt.figure(figsize=(14, 10))
sns.heatmap(
    cm_top,
    annot=False,
    fmt="d",
    xticklabels=top_names,
    yticklabels=top_names,
    cbar_kws={"label": "Count"}
)
plt.title(f"Confusion Matrix (Top {TOP_N} Classes in Val) - DistilBERT")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 5) Top confusion pairs (off-diagonal) across ALL labels
cm_all = confusion_matrix(val_true, val_preds, labels=labels_all)
errors = cm_all.copy()
np.fill_diagonal(errors, 0)

flat_idx = np.argsort(errors.ravel())[::-1]
shown = 0
TOP_K_ERRORS = 10

print("\n" + "="*60)
print(f"Top {TOP_K_ERRORS} Most Frequent Confusions (All Classes)")
print("="*60)

for idx in flat_idx:
    if shown >= TOP_K_ERRORS:
        break
    i, j = np.unravel_index(idx, errors.shape)
    cnt = errors[i, j]
    if cnt == 0:
        break
    print(f"{shown+1}. Actual: {le.classes_[i]}")
    print(f"   Pred:   {le.classes_[j]}")
    print(f"   Count:  {cnt}\n")
    shown += 1

3.6 验证集的预测结果做成一张“可人工审查”的表：把真值/预测、是否正确、预测置信度（softmax 最大概率）都写进 df_val，然后把“高置信度但预测错”的样本挑出来展示，最后保存成 CSV 方便你下载/分享/做误差分析。

In [ ]:
df_val = df_val.copy()

df_val["true_label_id"] = val_true
df_val["pred_label_id"] = val_preds
df_val["true_label"] = le.inverse_transform(df_val["true_label_id"])
df_val["pred_label"] = le.inverse_transform(df_val["pred_label_id"])
df_val["correct"] = (df_val["true_label_id"] == df_val["pred_label_id"])

# Confidence = max softmax prob
probs = torch.softmax(torch.tensor(pred_out.predictions), dim=-1).numpy()
df_val["pred_confidence"] = probs.max(axis=1)

print("Validation accuracy (recomputed):", df_val["correct"].mean())

# Show high-confidence mistakes (useful for discussion section)
worst = df_val[df_val["correct"] == False].sort_values("pred_confidence", ascending=False)
display(worst[[TEXT_COL, LABEL_COL, "true_label", "pred_label", "pred_confidence"]].head(20))

OUT_CSV = "val_predictions_distilbert.csv"
df_val.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

3.7 单条文本的推理（inference）：给一个新闻标题，跑你训练好的 DistilBERT 分类模型，输出 Top-K 最可能的类别及其概率。

In [ ]:
@torch.no_grad()
def predict_top_k(title: str, top_k: int = 5):
    model.eval()
    enc = tokenizer(
        str(title),
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt"
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model(**enc)
    probs = torch.softmax(out.logits, dim=-1).squeeze(0).cpu().numpy()

    top_idx = probs.argsort()[::-1][:top_k]
    return [(le.classes_[i], float(probs[i])) for i in top_idx]

test_title = "New AI technology improves medical diagnosis accuracy"
print("Input:", test_title)
for lbl, p in predict_top_k(test_title, top_k=5):
    print(f"{lbl}: {p*100:.2f}%")

In [ ]:
print(type(df_val), df_val.shape)
print(len(y_val_int), len(val_dataset))

print(type(df_val), df_val.shape)
print("y_val_int length:", len(y_val_int))
print("val_dataset length:", len(val_dataset))

4.1 Network Analysis

In [ ]:
# 按行为类别分析情绪变化
behavior_categories = {
    'professional': ['work', 'job', 'economy', 'career'],
    'educational': ['learning', 'education', 'knowledge', 'skill'],
    'social': ['society', 'relationship', 'interaction', 'community'],
    'ethical': ['ethics', 'safety', 'risk', 'responsibility']
}

for category, keywords in behavior_categories.items():
    mask = df['classes_str'].apply(lambda x: any(kw in x.lower() for kw in keywords))
    category_sentiment = df[mask]['sentiment_score']
    print(f"{category}: mean sentiment = {category_sentiment.mean():.3f}")

In [ ]:
# Analyze co-occurrence relationships between topics
import networkx as nx
from itertools import combinations
# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

# Function to extract noun phrases
def extract_noun_phrases(text):
    doc = nlp(text)
    # Extract noun phrases and convert to lowercase
    phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks]
    # Filter out phrases that are too short or contain only punctuation
    phrases = [p for p in phrases if len(p) > 2 and any(c.isalnum() for c in p)]
    return phrases

# Apply function to extract noun phrases
df["noun_phrases"] = df["classes_str"].apply(extract_noun_phrases)

# Calculate top 30 most frequent noun phrases
all_phrases = [phrase for sublist in df["noun_phrases"] for phrase in sublist]
phrase_counts = Counter(all_phrases)
top_30_phrases = [phrase for phrase, count in phrase_counts.most_common(30)]

print("Top 30 most frequent noun phrases:")
for i, (phrase, count) in enumerate(phrase_counts.most_common(30), 1):
    print(f"{i}. {phrase}: {count} times")

# Build topic co-occurrence network
G = nx.Graph()

# Add nodes
for phrase in top_30_phrases:
    G.add_node(phrase, size=phrase_counts[phrase])

# Add edges (co-occurrence relationships)
for phrases in df["noun_phrases"]:
    # Only take phrases from the top 30 topics
    main_phrases = [p for p in phrases if p in top_30_phrases]
    # If the article has 2 or more main phrases, add edges between them
    if len(main_phrases) >= 2:
        for pair in combinations(main_phrases, 2):
            if G.has_edge(*pair):
                G[pair[0]][pair[1]]["weight"] += 1
            else:
                G.add_edge(pair[0], pair[1], weight=1)

# Calculate network statistics
print(f"\nBasic network information:")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

# Calculate centrality metrics
betweenness = nx.betweenness_centrality(G)
degree_centrality = nx.degree_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)

print("\nMost important themes based on betweenness centrality:")
for theme, centrality in sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{theme}: {centrality:.4f}")

print("\nMost important themes based on degree centrality:")
for theme, centrality in sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{theme}: {centrality:.4f}")

# Visualize network
plt.figure(figsize=(16, 12))

# Node size based on degree centrality
node_sizes = [degree_centrality[node] * 5000 for node in G.nodes()]
# Node color based on betweenness centrality
node_colors = [betweenness[node] * 100 for node in G.nodes()]
# Edge width based on weight
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]

# Draw network
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, 
                       node_color=node_colors, 
                       cmap=plt.cm.plasma, alpha=0.8)

# Draw edges
nx.draw_networkx_edges(G, pos, width=[w/5 for w in edge_weights], 
                       alpha=0.5, edge_color='gray')

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

plt.title("AI Topic Co-occurrence Network", fontsize=16)
plt.axis('off')

# Add legend description
plt.figtext(0.5, 0.01, 
            "Node size represents degree centrality, node color represents betweenness centrality, edge width represents co-occurrence frequency",
            ha="center", fontsize=10, style='italic')

plt.tight_layout()
plt.show()

# Create subplots for more detailed analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Topic frequency bar chart
top_10_phrases = phrase_counts.most_common(10)
phrases, counts = zip(*top_10_phrases)
axes[0, 0].barh(range(len(phrases)), counts, color='skyblue')
axes[0, 0].set_yticks(range(len(phrases)))
axes[0, 0].set_yticklabels(phrases)
axes[0, 0].set_xlabel('Frequency')
axes[0, 0].set_title('Top 10 Topic Frequencies')
axes[0, 0].invert_yaxis()

# 2. Centrality metrics comparison
top_10_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
themes, btwn_values = zip(*top_10_betweenness)
axes[0, 1].bar(range(len(themes)), btwn_values, color='lightcoral')
axes[0, 1].set_xticks(range(len(themes)))
axes[0, 1].set_xticklabels(themes, rotation=45, ha='right')
axes[0, 1].set_ylabel('Betweenness Centrality')
axes[0, 1].set_title('Top 10 Themes by Betweenness Centrality')

# 3. Degree distribution histogram
degrees = [degree for node, degree in G.degree()]
axes[1, 0].hist(degrees, bins=15, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Degree')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Network Degree Distribution')
axes[1, 0].grid(True, alpha=0.3)

# 4. Community detection
from networkx.algorithms import community

# Use Louvain algorithm for community detection
try:
    communities = community.louvain_communities(G, weight='weight', seed=42)
    
    # Assign community colors to each node
    community_colors = {}
    for i, comm in enumerate(communities):
        for node in comm:
            community_colors[node] = i
    
    # Draw network colored by community
    node_colors_comm = [community_colors[node] for node in G.nodes()]
    
    pos_comm = nx.spring_layout(G, seed=42)
    nx.draw_networkx_nodes(G, pos_comm, node_size=300, 
                          node_color=node_colors_comm, 
                          cmap=plt.cm.tab20, alpha=0.8, ax=axes[1, 1])
    nx.draw_networkx_edges(G, pos_comm, alpha=0.3, ax=axes[1, 1])
    nx.draw_networkx_labels(G, pos_comm, font_size=8, ax=axes[1, 1])
    axes[1, 1].set_title(f'Community Structure (Detected {len(communities)} communities)')
    axes[1, 1].axis('off')
    
    # Print community information
    print(f"\nDetected {len(communities)} communities:")
    for i, comm in enumerate(communities):
        print(f"Community {i+1}: {list(comm)[:5]}... ({len(comm)} themes)")
        
except Exception as e:
    axes[1, 1].text(0.5, 0.5, f"Community detection failed: {str(e)}", 
                    ha='center', va='center', transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('Community Structure')
    axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Create topic clustering heatmap
# Create topic-topic co-occurrence matrix
topics = top_30_phrases
cooccurrence_matrix = np.zeros((len(topics), len(topics)))

for i, topic1 in enumerate(topics):
    for j, topic2 in enumerate(topics):
        if i != j and G.has_edge(topic1, topic2):
            cooccurrence_matrix[i, j] = G[topic1][topic2]['weight']

# Create heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(cooccurrence_matrix, 
            xticklabels=topics, 
            yticklabels=topics,
            cmap='YlOrRd',
            square=True,
            linewidths=0.5)
plt.title('Topic Co-occurrence Matrix Heatmap', fontsize=16)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Output most important topic connections
print("\nMost important topic connections (top 10 edges by weight):")
edges_with_weights = [(u, v, G[u][v]['weight']) for u, v in G.edges()]
edges_with_weights.sort(key=lambda x: x[2], reverse=True)
for i, (u, v, w) in enumerate(edges_with_weights[:10], 1):
print(f"{i}. {u} ↔ {v}: {w} co-occurrences")